In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from plotly.subplots import make_subplots
import plotly.graph_objs as go
import json
from datetime import datetime
from IPython.display import IFrame
import plotly.express as px
from scipy.interpolate import interp1d
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import json



# Cargar el dataset procesado de 100Hz
file_path = "zx6r_20240317_catalunya_procesado_100Hz.csv"
df = pd.read_csv(file_path)

# Mostrar las primeras filas y las columnas para revisar su estructura
df.head(), df.columns

In [ ]:
# Calculamos los estadísticos por vuelta
stats_per_lap = df.groupby("Lap_Number").agg(
    lap_time = ('Time', 'max'),
    speed_max = ('Speed', 'max'),
    speed_min=('Speed', 'min'),
    speed_mean=('Speed', 'mean'),
    accel_max=('Acceleration', 'max'),
    decel_max=('Acceleration', 'min'),
    pct_accelerating=('Acceleration_Sign', lambda x: (x == "Positive").mean() * 100),
    rpm_max=('Engine_RPM', 'max'),
    rpm_mean=('Engine_RPM', 'mean'),
    fuel_total=('Fuel_Consumed_Liters', 'max'),
    distance = ('Distance', 'max')
).reset_index()

# Redondeamos columnas específicas
stats_per_lap = stats_per_lap.round({
    'speed_max': 1,
    'speed_min': 1,
    'speed_mean': 1,
    'accel_max': 2,
    'decel_max': 2,
    'pct_accelerating': 0,
    'rpm_max': 0,
    'rpm_mean': 0,
    'fuel_total': 3,
    'distance':5
})

# Mostramos los resultados
stats_per_lap

In [ ]:
# Volver a calcular datos de delta sobre % de vuelta
distance_step = 0.01  # 10 metros
lap_numbers = sorted(df['Lap_Number'].unique())
interpolated_percent_laps = {}
percent_common = np.linspace(0, 1, int(1 / distance_step))  # puntos comunes en % de vuelta

# Identificar vuelta más rápida
lap_times = {lap: df[df['Lap_Number'] == lap]['Time'].max() for lap in lap_numbers}
reference_lap = min(lap_times, key=lap_times.get)
reference_lap_data = df[df['Lap_Number'] == reference_lap]
reference_distance = reference_lap_data['Distance'].max()
mapped_distance = percent_common * reference_distance

# Interpolar todas las vueltas sobre % de vuelta
for lap in lap_numbers:
    lap_data = df[df['Lap_Number'] == lap].copy()
    lap_distance = lap_data['Distance'].values
    lap_time = lap_data['Time'].values
    percent_distance = lap_distance / lap_distance[-1]  # normalizar de 0 a 1
    interp_time = np.interp(percent_common, percent_distance, lap_time)
    interpolated_percent_laps[lap] = interp_time

# Calcular deltas
reference_time = interpolated_percent_laps[reference_lap]
delta_df = pd.DataFrame({'Distance': mapped_distance})
for lap, time_interp in interpolated_percent_laps.items():
    if lap == reference_lap:
        delta_df[f'Lap_{lap}'] = 0
    else:
        delta_df[f'Lap_{lap}'] = time_interp - reference_time

delta_df.head(5)


In [ ]:
# CONFIGURACIÓN
colors = ['lime', 'cyan', 'magenta', 'red', 'yellow', 'orange']
lap_numbers = sorted(df['Lap_Number'].unique())
lap_colors = {lap: colors[i % len(colors)] for i, lap in enumerate(lap_numbers)}
reference_lap = lap_numbers[3]
reference_distance = df[df['Lap_Number'] == reference_lap]['Distance'].max()

samples_per_vuelta = 140 * 100  # 140 segundos * 100 Hz
percent_common = np.linspace(0, 1, samples_per_vuelta)
mapped_distance = percent_common * reference_distance

# INTERPOLACIÓN
speed_interp, accel_interp, gear_interp, accel_bin_interp = {}, {}, {}, {}
interpolated_percent_laps = {}

for lap in lap_numbers:
    lap_data = df[df['Lap_Number'] == lap].drop_duplicates(subset="Distance")

    f_speed = interp1d(lap_data['Distance'], lap_data['Speed'], bounds_error=False, fill_value="extrapolate")
    f_accel = interp1d(lap_data['Distance'], lap_data['Acceleration'], bounds_error=False, fill_value="extrapolate")
    f_gear = interp1d(lap_data['Distance'], lap_data['Gear'], bounds_error=False, fill_value="extrapolate")
    f_bin = interp1d(lap_data['Distance'], np.where(lap_data["Acceleration"] > 0, 1, -1),
                     bounds_error=False, fill_value="extrapolate")

    speed_interp[lap] = f_speed(mapped_distance)
    accel_interp[lap] = f_accel(mapped_distance)
    gear_interp[lap] = np.rint(f_gear(mapped_distance)).astype(int)
    accel_bin_interp[lap] = np.where(f_bin(mapped_distance) >=0, 1, -1)

    percent_distance = lap_data["Distance"] / lap_data["Distance"].max()
    f_time = interp1d(percent_distance, lap_data["Time"], bounds_error=False, fill_value="extrapolate")
    interpolated_percent_laps[lap] = f_time(percent_common)

# GRÁFICO
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Delta de tiempo respecto a la vuelta más rápida",
        "Velocidad por vuelta (km/h)",
        "Aceleración por vuelta (g)",
        "Marcha engranada",
        "Frenada [-1]/ Aceleración [1]"
    )
)

for lap in lap_numbers:
    fig.add_trace(go.Scatter(
        x=mapped_distance,
        y=interpolated_percent_laps[lap] - interpolated_percent_laps[reference_lap] if lap != reference_lap else [0]*len(mapped_distance),
        mode='lines',
        name=f"Lap {lap}",
        line=dict(color=lap_colors[lap]),
        hovertemplate='%{y:.2f} s'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(x=mapped_distance, y=speed_interp[lap], mode='lines',
                             name=f"Lap {lap}", line=dict(color=lap_colors[lap]), hovertemplate='%{y:.1f} km/h',
                             showlegend=False), row=2, col=1)

    fig.add_trace(go.Scatter(x=mapped_distance, y=accel_interp[lap], mode='lines',
                             name=f"Lap {lap}", line=dict(color=lap_colors[lap]), hovertemplate='%{y:.2f} g',
                             showlegend=False), row=3, col=1)

    fig.add_trace(go.Scatter(x=mapped_distance, y=gear_interp[lap], mode='lines',
                             name=f"Lap {lap}", line=dict(color=lap_colors[lap]), hovertemplate='%{y}',
                             showlegend=False), row=4, col=1)

    fig.add_trace(go.Scatter(x=mapped_distance, y=accel_bin_interp[lap], mode='lines',
                             name=f"Lap {lap}", line=dict(color=lap_colors[lap]), hovertemplate='%{y}',
                             showlegend=False), row=5, col=1)

fig.update_layout(
    hovermode="x",
    hoverlabel=dict(bgcolor="black", font_size=12, font_family="Arial"),
    template='plotly_dark',
    height=1200,
    width=1000,
    title_text="Análisis temporal por vuelta",
    title_x=0.5,
    title_font=dict(size=22, color='white'),
    legend=dict(x=1.02, y=1, bgcolor='rgba(0,0,0,0)', bordercolor='white')
)

# Etiquetas de ejes
fig.update_xaxes(title_text="Distancia escalada (km)", row=5, col=1)
fig.update_yaxes(title_text="Delta (s)", row=1, col=1)
fig.update_yaxes(title_text="Velocidad (km/h)", row=2, col=1)
fig.update_yaxes(title_text="Aceleración (g)", row=3, col=1)
fig.update_yaxes(title_text="Marcha", row=4, col=1)
fig.update_yaxes(title_text="Frenada / Aceleración", row=5, col=1)

# Mostrar
fig.show()


# Guardar HTML
html_path = "analisis_temporal.html"
fig.write_html(html_path)

In [ ]:
# Formato de tiempo
def format_lap_time(seconds):
    minutes = int(seconds // 60)
    remaining = seconds % 60
    return f"{minutes}:{remaining:06.3f}"

stats_per_lap["lap_time_num"] = stats_per_lap["lap_time"]
stats_per_lap["lap_time"] = stats_per_lap["lap_time"].astype(float).apply(format_lap_time)



# % de uso de marchas
lap_totals = df.groupby("Lap_Number").size()

# Conteo por marcha
gear_counts_check = df.groupby(["Lap_Number", "Gear"]).size().unstack(fill_value=0)
suma_por_marcha = gear_counts_check.sum(axis=1)

comparacion = pd.DataFrame({
    "Total por vuelta": lap_totals,
    "Suma de marchas": suma_por_marcha,
    "Coincide": lap_totals == suma_por_marcha
})
totals = df.groupby("Lap_Number").size()
gear_pct_by_lap = gear_counts_check.div(lap_totals, axis=0) * 100
gear_pct_by_lap = gear_pct_by_lap.round(1)

# Creación de JSON
js_stats_data = stats_per_lap.set_index("Lap_Number")[[
    "lap_time", "lap_time_num", "speed_max", "speed_min", "speed_mean",
    "accel_max", "decel_max", "pct_accelerating",
    "rpm_max", "rpm_mean", "fuel_total", "distance"
]].T.to_dict()

marchas_usadas = [1,2,3,4,5,6]
gear_data_json = {
    str(lap): [gear_pct_by_lap.loc[lap][g] if g in gear_pct_by_lap.columns else 0.0 for g in marchas_usadas]
    for lap in gear_pct_by_lap.index
}

# HTML
html_code = f"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Estadísticos por Vuelta</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    body {{
      background-color: black;
      color: white;
      font-family: Arial, sans-serif;
      padding: 20px;
    }}
    h1 {{ text-align: center; margin-bottom: 30px; }}
    .selectors {{ text-align: center; margin-bottom: 20px; }}
    .tables {{ display: flex; justify-content: space-around; gap: 40px; margin-bottom: 40px; }}
    table {{
      border-collapse: collapse;
      width: 300px;
      background-color: #111;
      border: 1px solid #666;
    }}
    th, td {{
      border: 1px solid #444;
      padding: 8px;
      text-align: center;
    }}
    th {{ background-color: #222; }}
  </style>
</head>
<body>
  <h1>Estadísticos por Vuelta</h1>
  <div class="selectors">
    <label>Vuelta A: <select id="lapA"></select></label>
    <label style="margin-left:20px;">Vuelta B: <select id="lapB"></select></label>
  </div>
  <div class="tables">
    <div><h3 style="text-align:center">Vuelta A</h3><table id="tablaA"></table></div>
    <div><h3 style="text-align:center">Vuelta B</h3><table id="tablaB"></table></div>
  </div>
  <div id="histograma" style="height: 500px;"></div>

  <script>
    const statsData = {json.dumps(js_stats_data, indent=2)};
    const gearData = {json.dumps(gear_data_json, indent=2)};
    const statLabels = {{
      "lap_time": "Tiempo vuelta (mm:ss.mmm)",
      "speed_max": "Velocidad máxima (km/h)",
      "speed_min": "Velocidad mínima (km/h)",
      "speed_mean": "Velocidad media (km/h)",
      "accel_max": "Aceleración máxima (g)",
      "decel_max": "Deceleración máxima (g)",
      "pct_accelerating": "% tiempo acelerando",
      "rpm_max": "RPM máximo",
      "rpm_mean": "RPM medio",
      "fuel_total": "Combustible consumido (L)",
      "distance": "Distancia vuelta (km)"
    }};

    const lapSelectA = document.getElementById("lapA");
    const lapSelectB = document.getElementById("lapB");
    const tablaA = document.getElementById("tablaA");
    const tablaB = document.getElementById("tablaB");

    for (let lap in statsData) {{
      lapSelectA.innerHTML += `<option value="${{lap}}">Lap ${{lap}}</option>`;
      lapSelectB.innerHTML += `<option value="${{lap}}">Lap ${{lap}}</option>`;
    }}

    lapSelectA.value = "4";
    lapSelectB.value = "6";

    function updateTablesAndChart() {{
      const lapA = lapSelectA.value;
      const lapB = lapSelectB.value;

      tablaA.innerHTML = "";
      tablaB.innerHTML = "";

      const menorEsMejor = ["lap_time", "decel_max", "fuel_total", "distance"];
      const mayorEsMejor = ["speed_max", "speed_mean", "accel_max", "pct_accelerating", "rpm_max", "rpm_mean"];

      for (let key in statLabels) {{
        const valShowA = statsData[lapA][key];
        const valShowB = statsData[lapB][key];

        const valCompA = statsData[lapA][key + "_num"] ?? parseFloat(valShowA);
        const valCompB = statsData[lapB][key + "_num"] ?? parseFloat(valShowB);

        let mejorA = false, mejorB = false;

        if (!isNaN(valCompA) && !isNaN(valCompB)) {{
          if (menorEsMejor.includes(key)) {{
            mejorA = valCompA < valCompB;
            mejorB = valCompB < valCompA;
          }} else if (mayorEsMejor.includes(key)) {{
            mejorA = valCompA > valCompB;
            mejorB = valCompB > valCompA;
          }}
        }}

        const highlightA = mejorA ? `<span style='color:gold;font-weight:bold;'>${{valShowA}}</span>` : valShowA;
        const highlightB = mejorB ? `<span style='color:gold;font-weight:bold;'>${{valShowB}}</span>` : valShowB;

        const rowA = tablaA.insertRow();
        rowA.insertCell().textContent = statLabels[key];
        rowA.insertCell().innerHTML = highlightA;

        const rowB = tablaB.insertRow();
        rowB.insertCell().textContent = statLabels[key];
        rowB.insertCell().innerHTML = highlightB;
      }}

      const gears = ["1", "2", "3", "4", "5", "6"];
      const traceA = {{
        x: gears,
        y: gearData[lapA],
        name: 'Lap ' + lapA,
        type: 'bar',
        marker: {{color: 'limegreen'}},
        orientation: 'v',
        offsetgroup: 0
      }};
      const traceB = {{
        x: gears,
        y: gearData[lapB],
        name: 'Lap ' + lapB,
        type: 'bar',
        marker: {{color: 'orangered'}},
        orientation: 'v',
        offsetgroup: 1
      }};
       Plotly.newPlot('histograma', [traceA, traceB], {{
        barmode: 'group',
        template: 'plotly_dark',
        paper_bgcolor: 'black',
        plot_bgcolor: 'black',
        title: {{ text: 'Distribución de marchas (%)', font: {{ color: 'white' }} }},
        xaxis: {{
          title: 'Marcha',
          color: 'white',
          tickfont: {{ color: 'white' }},
          titlefont: {{ color: 'white' }},
          gridcolor: 'white'
        }},
        yaxis: {{
          title: 'Porcentaje (%)',
          color: 'white',
          tickfont: {{ color: 'white' }},
          titlefont: {{ color: 'white' }},
          gridcolor: 'white'
        }}
      }});
    }}

    lapSelectA.addEventListener("change", updateTablesAndChart);
    lapSelectB.addEventListener("change", updateTablesAndChart);
    updateTablesAndChart();
  </script>
</body>
</html>
"""

# Guardamos
with open("comparador_estadisticos_completo_final.html", "w", encoding="utf-8") as f:
    f.write(html_code)
# Mostramos el gráfico
IFrame(src="comparador_estadisticos_completo_final.html", width=1000, height=800)


In [ ]:
# Generamos el mapa

In [ ]:
# Variables de interés
variables = ["Speed", "Gear", "Acceleration_State", "Acceleration", "Engine_RPM"]
df["Acceleration_State"] = np.where(df["Acceleration"] >= 0, 1, -1)
lap_dict = {}
df_map = df[["Speed", "Gear", "Acceleration_State", "Lap_Number", "Latitude", "Longitude", "Acceleration", "Engine_RPM"]]
for lap, group in df_map.groupby("Lap_Number"):
    lap_dict[int(lap)] = group.to_dict(orient="records")

# JSON
json_data = json.dumps(lap_dict)

# HTML
html_code = f"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8" />
  <title>Mapa Comparativo</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <style>
    body {{
      background-color: black;
      color: white;
      font-family: Arial, sans-serif;
      margin: 0;
      padding: 0;
    }}
    h1 {{
      text-align: center;
      margin-top: 20px;
    }}
    .controls {{
      text-align: center;
      margin: 10px 0;
    }}
    select {{
      padding: 6px;
      font-size: 14px;
      background-color: #222;
      color: white;
      border: 1px solid #555;
    }}
    .layout-container {{
      display: flex;
      justify-content: space-between;
      align-items: center;
      padding: 0 10px;
    }}
    .legend-container {{
      width: 80px;
      height: 600px;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: space-between;
    }}
    .legend-bar {{
      width: 20px;
      height: 400px;
      border: 1px solid white;
    }}
    .legend-label {{
      font-size: 12px;
      text-align: center;
      color: white;
    }}
    .plot-container {{
      flex-grow: 1;
    }}
  </style>
</head>
<body>
  <h1>Mapa Comparativo</h1>
  <div class="controls">
    <label for="lapA">Vuelta A:</label>
    <select id="lapA"></select>
    <label for="lapB">Vuelta B:</label>
    <select id="lapB"></select>
    <label for="variable">Variable:</label>
    <select id="variable">
      {''.join(f'<option value="{v}">{v}</option>' for v in variables)}
    </select>
  </div>
  <div class="layout-container">
    <div class="legend-container">
      <div class="legend-label" id="labelA">Lap A</div>
      <div class="legend-label" id="minA">Min</div>
      <div class="legend-bar" id="barA" style="background: linear-gradient(to top, rgb(150,0,0), white);"></div>
      <div class="legend-label" id="maxA">Max</div>
    </div>
    <div class="plot-container">
      <div id="mapa" style="height: 600px;"></div>
    </div>
    <div class="legend-container">
      <div class="legend-label" id="labelB">Lap B</div>
      <div class="legend-label" id="minB">Min</div>
      <div class="legend-bar" id="barB" style="background: linear-gradient(to top, rgb(0,150,0), white);"></div>
      <div class="legend-label" id="maxB">Max</div>
    </div>
  </div>

  <script>
    const rawData = {json_data};

    const lapSelectA = document.getElementById("lapA");
    const lapSelectB = document.getElementById("lapB");
    const allLaps = Object.keys(rawData);

    for (let lap of allLaps) {{
      lapSelectA.innerHTML += `<option value="${{lap}}">Lap ${{lap}}</option>`;
      lapSelectB.innerHTML += `<option value="${{lap}}">Lap ${{lap}}</option>`;
    }}
    lapSelectA.value = allLaps[0];
    lapSelectB.value = allLaps[1];

    document.getElementById("lapA").addEventListener("change", updateMap);
    document.getElementById("lapB").addEventListener("change", updateMap);
    document.getElementById("variable").addEventListener("change", updateMap);

    function updateMap() {{
      const lapA = lapSelectA.value;
      const lapB = lapSelectB.value;
      const variable = document.getElementById("variable").value;

      const pointsA = rawData[lapA];
      const pointsB = rawData[lapB];

      const allValues = pointsA.concat(pointsB).map(p => +p[variable]);
      const vmin = Math.min(...allValues);
      const vmax = Math.max(...allValues);

      document.getElementById("minA").innerText = vmin.toFixed(1);
      document.getElementById("maxA").innerText = vmax.toFixed(1);
      document.getElementById("minB").innerText = vmin.toFixed(1);
      document.getElementById("maxB").innerText = vmax.toFixed(1);
      document.getElementById("labelA").innerText = "Lap " + lapA;
      document.getElementById("labelB").innerText = "Lap " + lapB;

      const traceA = {{
        type: 'scattermapbox',
        mode: 'markers',
        lat: pointsA.map(p => p["Latitude"]),
        lon: pointsA.map(p => p["Longitude"]),
        marker: {{
          size: 6,
          color: pointsA.map(p => +p[variable]),
          colorscale: 'Reds',
          cmin: vmin,
          cmax: vmax,
          showscale: false
        }},
        text: pointsA.map(p => `Lap ${{lapA}}<br>${{variable}}: ${{p[variable]}}`),
        hoverinfo: 'text',
        showlegend: false
      }};

      const traceB = {{
        type: 'scattermapbox',
        mode: 'markers',
        lat: pointsB.map(p => p["Latitude"]),
        lon: pointsB.map(p => p["Longitude"]),
        marker: {{
          size: 6,
          color: pointsB.map(p => +p[variable]),
          colorscale: 'Greens',
          reversescale: true,
          cmin: vmin,
          cmax: vmax,
          showscale: false
        }},
        text: pointsB.map(p => `Lap ${{lapB}}<br>${{variable}}: ${{p[variable]}}`),
        hoverinfo: 'text',
        showlegend: false
      }};

      Plotly.newPlot("mapa", [traceA, traceB], {{
        mapbox: {{
          style: "carto-darkmatter",
          center: {{
            lat: pointsA[0]["Latitude"],
            lon: pointsA[0]["Longitude"]
          }},
          zoom: 15
        }},
        margin: {{ t: 0, b: 0, l: 0, r: 0 }},
        height: 600,
        template: "plotly_dark"
      }});
    }}

    updateMap();
  </script>
</body>
</html>
"""

# Guardamos
html_path = "mapa_comparador_final_estetico.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_code)
# Mostramos
IFrame(src="mapa_comparador_final_estetico.html", width=1000, height=800)